In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-employee-burnout-score-using-machine-learning/test_Kaggle_Comp.csv
/kaggle/input/predicting-employee-burnout-score-using-machine-learning/train_Kaggle_Comp.csv
/kaggle/input/predicting-employee-burnout-score-using-machine-learning/sample_submission_Kaggle_Comp.csv


In [2]:
# ==========================================================
# FINAL STABLE & OPTIMIZED KAGGLE PIPELINE (NO API ERRORS)
# Models: LightGBM + CatBoost + ExtraTrees + Stacking
# ==========================================================

import numpy as np
import pandas as pd
import gc
import warnings

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesRegressor

import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
RANDOM_SEED = 42

# ==========================================================
# 1. LOAD DATA
# ==========================================================
TRAIN_PATH = '/kaggle/input/predicting-employee-burnout-score-using-machine-learning/train_Kaggle_Comp.csv'
TEST_PATH  = '/kaggle/input/predicting-employee-burnout-score-using-machine-learning/test_Kaggle_Comp.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

TARGET = 'burnout_score'
ID_COL = 'employee_id'

y = train[TARGET].values
test_ids = test[ID_COL]

# ==========================================================
# 2. FEATURE ENGINEERING
# ==========================================================
def engineer(df):
    df['total_work_load'] = df['working_hours_per_day'] + df['overtime_hours']
    df['pressure_sleep_ratio'] = df['work_pressure_score'] / (df['sleep_hours'] + 1)
    df['stress_gap'] = df['stress_level'] - df['job_satisfaction']
    df['fatigue_pressure'] = df['mental_fatigue_score'] * df['work_pressure_score']
    df['support_balance'] = df['manager_support'] - df['work_pressure_score']
    return df

train = engineer(train)
test  = engineer(test)

# ==========================================================
# 3. SAFE TARGET ENCODING
# ==========================================================
def target_encode(train_df, test_df, col, target):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean','count'])
    smooth = (stats['count'] * stats['mean'] + 20 * global_mean) / (stats['count'] + 20)
    train_df[col + '_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[col + '_te'] = test_df[col].map(smooth).fillna(global_mean)

te_cols = ['job_role', 'company_type', 'deadline_frequency']
for col in te_cols:
    target_encode(train, test, col, TARGET)

# ==========================================================
# 4. LABEL ENCODING (SAFE)
# ==========================================================
cat_cols = ['gender','education_level','job_role','company_type',
            'deadline_frequency','remote_work','physical_activity','health_issues']

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# ==========================================================
# 5. FINAL FEATURES
# ==========================================================
drop_cols = [TARGET, ID_COL]
features = [c for c in train.columns if c not in drop_cols]

X = train[features]
X_test = test[features]

# ==========================================================
# 6. K-FOLD TRAINING
# ==========================================================
NFOLDS = 5
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
oof_et  = np.zeros(len(train))

pred_lgb = np.zeros(len(test))
pred_cat = np.zeros(len(test))
pred_et  = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold+1}")

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    # ---------- LightGBM ----------
    lgb_model = lgb.LGBMRegressor(
        n_estimators=6000,
        learning_rate=0.01,
        num_leaves=63,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(200)])
    oof_lgb[val_idx] = lgb_model.predict(X_val)
    pred_lgb += lgb_model.predict(X_test) / NFOLDS

    # ---------- CatBoost ----------
    cat_model = CatBoostRegressor(
        iterations=6000,
        learning_rate=0.01,
        depth=8,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        verbose=False
    )
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=200)
    oof_cat[val_idx] = cat_model.predict(X_val)
    pred_cat += cat_model.predict(X_test) / NFOLDS

    # ---------- ExtraTrees ----------
    et_model = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=5,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    et_model.fit(X_tr, y_tr)
    oof_et[val_idx] = et_model.predict(X_val)
    pred_et += et_model.predict(X_test) / NFOLDS

# ==========================================================
# 7. STACKING
# ==========================================================
meta_X = np.column_stack([oof_lgb, oof_cat, oof_et])
meta_test = np.column_stack([pred_lgb, pred_cat, pred_et])

meta_model = Ridge(alpha=1.0)
meta_model.fit(meta_X, y)

stacked_preds = meta_model.predict(meta_test)

# ==========================================================
# 8. FINAL BLEND + CLIP
# ==========================================================
final_preds = (
    0.6 * stacked_preds +
    0.25 * pred_cat +
    0.15 * pred_lgb
)

final_preds = np.clip(final_preds, 0.0, 1.0)

# ==========================================================
# 9. SUBMISSION
# ==========================================================
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_preds
})

submission.to_csv('submission.csv', index=False)
print("submission.csv saved")

# ==========================================================
# 10. CV METRICS
# ==========================================================
rmse = lambda y, p: np.sqrt(mean_squared_error(y, p))
print("\nCV RMSE:")
print("LightGBM:", rmse(y, oof_lgb))
print("CatBoost:", rmse(y, oof_cat))
print("ExtraTrees:", rmse(y, oof_et))
print("Stacked :", rmse(y, meta_model.predict(meta_X)))


Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001831 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 906
[LightGBM] [Info] Number of data points in the train set: 3200, number of used features: 29
[LightGBM] [Info] Start training from score 0.451726
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L